In [1]:
#loading the dataset
import pandas as pd
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")
movies = movies.merge(credits , on="title")
movies.shape

(4809, 23)

In [3]:
movies.head

<bound method NDFrame.head of          budget                                             genres  \
0     237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1     300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2     245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3     250000000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4     260000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
...         ...                                                ...   
4804     220000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4805       9000  [{"id": 35, "name": "Comedy"}, {"id": 10749, "...   
4806          0  [{"id": 35, "name": "Comedy"}, {"id": 18, "nam...   
4807          0                                                 []   
4808          0                [{"id": 99, "name": "Documentary"}]   

                                               homepage      id  \
0                           http://www.avatarmovie.com/   1999

In [4]:
#exploring data 
movies = movies [["movie_id","title","overview","genres","keywords","cast","crew"]]
movies.head

<bound method NDFrame.head of       movie_id                                     title  \
0        19995                                    Avatar   
1          285  Pirates of the Caribbean: At World's End   
2       206647                                   Spectre   
3        49026                     The Dark Knight Rises   
4        49529                               John Carter   
...        ...                                       ...   
4804      9367                               El Mariachi   
4805     72766                                 Newlyweds   
4806    231617                 Signed, Sealed, Delivered   
4807    126186                          Shanghai Calling   
4808     25975                         My Date with Drew   

                                               overview  \
0     In the 22nd century, a paraplegic Marine is di...   
1     Captain Barbossa, long believed to be dead, ha...   
2     A cryptic message from Bond’s past sends him o...   
3     Followi

In [5]:
#preprocesing 
movies = movies.dropna()
movies = movies.drop_duplicates()
movies = movies.reset_index(drop=True)
len(movies)


4806

In [6]:
import ast 
def names(text):
    result = []

    for d in ast.literal_eval(text):
        result.append(d["name"])

    return result
def director(text):
    return [
        d["name"]
        for d in ast.literal_eval(text)
        if d["job"] == "Director"
    ]

movies["genres"]=movies["genres"].apply(names)
movies["keywords"]=movies["keywords"].apply(names)
movies["cast"]=movies["cast"].apply(
  lambda x : names(x)[:3] #only the top 3  actors using the anonymous function lambda 
)
movies["crew"]=movies["crew"].apply(director)
movies.iloc[0]
  


  

movie_id                                                19995
title                                                  Avatar
overview    In the 22nd century, a paraplegic Marine is di...
genres          [Action, Adventure, Fantasy, Science Fiction]
keywords    [culture clash, future, space war, space colon...
cast         [Sam Worthington, Zoe Saldana, Sigourney Weaver]
crew                                          [James Cameron]
Name: 0, dtype: object

In [7]:
#prepare the data / one coulmn for tags 
movies["overview"]=movies["overview"].apply(
  lambda x : x.split())
for col in ["genres","keywords","cast","crew"]:
  movies[col]=movies[col].apply(
    lambda x:[i.replace(" ","")for i in x]
  )
movies["tags"] = (movies["overview"]+movies["genres"]+movies["keywords"]+movies["cast"]+movies["crew"])

In [8]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
#lowercase and stemming 
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
movies["tags"]=movies["tags"].apply(
  lambda x : " ".join(x).lower()
)
movies["tags"]=movies["tags"].apply(
  lambda x : " ".join(ps.stem(w) for w in x.split())
)
print(ps.stem("loving"),ps.stem("loved"))

love love


In [11]:
#vectorize
from sklearn.feature_extraction.text import (CountVectorizer)
cv = CountVectorizer(max_features=5000,stop_words="english")
vectors = cv.fit_transform(movies["tags"]).toarray()
vectors.shape

(4806, 5000)

In [12]:
#similarity
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)
similarity.shape

(4806, 4806)

In [13]:
#the model
def recommend(movie):
  index = movies[movies["title"]==movie].index[0]
  scores = list(enumerate(similarity[index]))
  scores = sorted(scores, key=lambda x :x[1],reverse = True)
  for i , score in scores[1:6]:
    print(movies.iloc[i].title)

In [15]:
recommend("The Dark Knight")

The Dark Knight Rises
Batman Begins
Batman Returns
Batman Forever
Batman


In [16]:
recommend("Inception")

12 Rounds
RED
Abduction
Krrish
The Animal


In [18]:
recommend("Avatar")

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.
